<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/ALIGNN_example_AIMS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atomistic Line Graph Neural Network (ALIGNN)

The ALIGNN framework (reference [paper](https://www.nature.com/articles/s41524-021-00650-1/)) is used to encode crystal structure information in graph representations which then interfaces with a message passing neural network. This neural network framework can update the embeddings of nodes and edges in the graph while retaining graph connectivity and allowing neighboring nodes and edges to exchange information (pass messages) about their state.

# Table of contents

1. Installing [ALIGNN](https://github.com/usnistgov/alignn)
2. Training [JARVIS-DFT](https://jarvis.nist.gov/jarvisdft) 2D exfoliation energy model 
5. Using trained models to make fast predictions for compounds with unknown exfoliation energies

In [ ]:
!pip install alignn

In [ ]:
import os
!pwd
os.chdir('/content')
# Clone ALIGNN repo to get example folder
if not os.path.exists('alignn'):
  !git clone https://github.com/usnistgov/alignn.git

os.chdir('alignn')
# Install using setup.py in case pip didn't work
# !python setup.py develop

#!pip install dgl-cu111 # Colab has cuda 11.1

#Train a model for JARVIS-DFT Exfoliation energy

**Get data in id_prop.csv format**

Here, we will organize the data used for training and validation into the appropriate format to train the ALIGNN model. 

The data is collected here from the Jarvis-DFT 3D dataset, which contains the relaxed structures and computed material properties for over 55,000 materials. 

The training data should be formatted as a simple CSV file listing the structure file for the sample (e.g. POSCAR-JVASP-10.vasp) and then the target property value. Example format:

``POSCAR-JVASP-10.vasp,71.12950718943668``

``POSCAR-JVASP-1372.vasp,44.54199597616097``  

``POSCAR-JVASP-14873.vasp,110.43563937368668``

**Try modifying**

Change ``prop`` from ``exfoliation energy`` to another property in the JARVIS-DFT 3D database. 

In [ ]:
from jarvis.db.figshare import data as jdata
from jarvis.core.atoms import Atoms
import os

cwd = os.getcwd() #current working directory
temp_dir_name = "DataDir_ExfoEnergy" 

if not os.path.isdir(temp_dir_name):
  os.makedirs(temp_dir_name)
os.chdir(temp_dir_name)

#Load JARVIS-DFT 3D Dataset
dft_3d = jdata("dft_3d")

#List all properties for first entry
dft_3d[0].keys()


In [ ]:
#Select property (use different property tag to change prediction)
prop = "exfoliation_energy" #"optb88vdw_bandgap"
f = open("id_prop.csv", "w")

'''
Check each entry in JARVIS-DFT 3D dataset to see if the exfoliation energy is 
defined. If so, include in the id_props dataset. Additionally, write out the 
POSCAR structure file for each entry.
'''
for i in dft_3d:
    atoms = Atoms.from_dict(i["atoms"])
    jid = i["jid"]
    poscar_name = "POSCAR-" + jid + ".vasp"
    target = i[prop]
    if target != "na":
        atoms.write_poscar(poscar_name)
        f.write("%s,%6f\n" % (poscar_name, target))
f.close()

os.chdir(cwd)

In [ ]:
#Count how many POSCAR files are in the current directory
!ls -altr  DataDir_ExfoEnergy/*.vasp | wc -l


In [ ]:
 #Count how many lines are in the id_prop.csv file (should match value above)
 !wc -l DataDir_ExfoEnergy/id_prop.csv 

**run train_folder.py**

In addition to the dataset file (id_prop.csv) a configuration file to describe the essential parameters of the atomistic line graph neural network should be located in the same directory. The configuration file should be formatted as a ``json`` file, and an example file can be found [here](https://github.com/usnistgov/alignn/blob/main/alignn/examples/sample_data/config_example.json).

**Try modifying**

Common attributes to change from default setting may include:

1) Number of epochs: number of cycles used to train the neural network using the entire training set

2) Batch size: number of samples used to train the neural network at once

3) Output features: default is 1 for predicting single, scalar values; adjust if predicting a multi-output quantity

4) ALIGNN layers/ GCN layers: number of ALIGNN update layers or edge-gated graph convolution layers

In [ ]:
import time
t1=time.time()
!train_folder.py --root_dir "DataDir_ExfoEnergy" --epochs 50 --batch_size 64 --config "alignn/examples/sample_data/config_example.json" --output_dir="ExfoEnOut"
t2=time.time()
print ('Time in s',t2-t1)

In [ ]:
!ls ExfoEnOut	 

Plot out the "training curves" for the neural network, or the loss as a function of training epochs for both the training and validation curves. A properly trained model should have low loss values for both datasets.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import json
with open('ExfoEnOut/history_train.json') as ftrain:
  train_trace = json.load(ftrain)

with open('ExfoEnOut/history_val.json') as fval:
  val_trace = json.load(fval)

epochs = np.arange(1, 51)
plt.figure()

plt.plot(epochs, val_trace['mae'], label = 'Validation', color='xkcd:black')

plt.plot(epochs, train_trace['mae'], label = 'Training', color = 'xkcd:red')

plt.ylabel('MAE')
plt.xlabel('Epochs')
plt.legend()

Here checkpoints are the model parameter files that can be loaded in torch library to make predictions such as [this example](https://github.com/usnistgov/alignn/blob/main/alignn/scripts/predict.py). The checkpoint files can be used to run the trained model on new samples.

In [ ]:
import pandas as pd
df = pd.read_csv('/content/alignn/ExfoEnOut/prediction_results_test_set.csv')

These are the predictions for the 10% of the dataset that the model did not see during training and validaiton steps.

In [ ]:
df

In [ ]:
plt.plot(df['target'],df['prediction'],'.')
plt.plot(df['target'],df['target'],'-.')
plt.xlabel('DFT data (meV)')
plt.ylabel('ALIGNN predictions (meV)')
plt.show()

**run model just trained on new samples without exfoliation energies listed**

1. Load the model parameters saved in the final checkpoint file 
2. Fetch the features for 50 samples which do not have exfoliation energies listed on the JARVIS-DFT 3D dataset
3. Run model for prediction on multiple samples using ``get_multiple_predictions``

In [ ]:
# Run Trained Model to Predict Values for Compounds without Exfoliation Energy

import torch
from alignn.pretrained import get_multiple_predictions
from alignn.models.alignn import ALIGNN,ALIGNNConfig

if torch.cuda.is_available():
    device = torch.device("cuda") # might have to change to "cpu"


model = ALIGNN(ALIGNNConfig(name="alignn"))
model.load_state_dict(torch.load('ExfoEnOut/checkpoint_50.pt', map_location=device)["model"])
model.to(device)
model.eval()
atoms_array=[]

max_samples = 50
count = 1
for i in dft_3d:
    atoms = Atoms.from_dict(i["atoms"])
    jid = i["jid"]
    poscar_name = "POSCAR-" + jid + ".vasp"
    target = i[prop]
    if target == "na":
        atoms_array.append(atoms)
        count = count + 1
        if count == max_samples:
          break

get_multiple_predictions(atoms_array=atoms_array,model=model)

These are the exfoliation energy predictions for unlabelled compositions in the JARVIS-DFT 3D dataset.

In [ ]:
import pandas as pd
 
pred_df=pd.read_json('pred_data.json')

#Add the chemical formula as a column in the prediction dataframe
cf_list = []
for index, row in pred_df.iterrows():
  atoms = Atoms.from_dict(row["atoms"])
  cf_list.append(atoms.composition.reduced_formula)

pred_df['formula'] = cf_list
pred_df.sort_values('pred',ascending=False)[:10]